# M12-HF — Realized Volatility haute-fréquence (minute) pour HAR-RV-J

**Question de recherche.** Le modele HAR-RV-J deploye (M12, [Andersen-Bollerslev-Diebold 2007](https://doi.org/10.1162/rest.89.4.701)) estime la realized variance (RV), la bipower variation (BPV) et la composante de saut (J) a partir de rendements **intraday horaires** (24 obs/jour en crypto 24/7). La theorie ([ABDL 2003](https://doi.org/10.1111/1468-0262.00418)) predit que la RV converge vers la variance integree quand la frequence d'echantillonnage augmente : un passage **horaire -> minute** (1440 obs/jour) devrait resserrer l'estimateur et nettoyer le signal de saut.

**Hypothese testable (a falsifier).** Si la RV/BPV minute est plus precise, les composantes de saut $J_t = \max(RV_t - \mu \cdot BPV_t, 0)$ ($\mu \approx 0.6$, Huang-Tauchen) deviennent plus nettes, et le HAR-RV-J **minute** (M12-HF) devrait battre le HAR-RV-J **horaire** (M12, le KEEPER actuel) sur le Sharpe de Kelly — surtout a horizon court $h=1$.

**Risque documente.** Sous ~1-5 min, le bruit de microstructure (bid-ask bounce) peut *influer* la RV ([Hansen-Lunde 2006](https://doi.org/10.1198/073500106000000072)). M12-HF pourrait donc **ne pas battre** M12-horaire sur les coins peu liquides (DOT, ADA) — ce serait deja un resultat utile (cadre de validite de la frequence d'echantillonnage).

**Comparatif decisif = M12-horaire** (re-calcule ici meme sur donnee Cloud horaire gratuite, meme environnement, meme fenetre, pour une comparaison equitable). **Donnee minute crypto Binance = Cloud Access gratuit sur QC** (0 QCC, aucun `lean data download`, aucun achat) — feu vert user 2026-06-24. Spec complete : [`docs/M12_HF_PROPOSAL.md`](docs/M12_HF_PROPOSAL.md).

> Ce notebook QuantBook s'execute sur QC Cloud Research (data minute gratuite). Il est **auto-contenu** : estimateurs RV/BPV/J, modeles HAR / HAR-RV-J, walk-forward 5-fold, sizing Kelly, tests DM + sign-test sont inlines (l'environnement Research QC est isole, pas d'import des modules `scripts/` locaux). **Pipeline per-coin incrémental + chunking annuel** : la data minute (~4M bars/coin/an) est tiree et agreggee annee par annee puis liberee, pour rester sous la limite memoire du nœud Research (sinon OOM).

## 1. Setup QuantBook — 7 coins, marche Binance

Univers anti-FAANG : BTC, ETH, SOL, LTC, XRP, ADA, DOT (cf M12 / `CURRICULUM.md` Stage 3a). Marche **Binance** (liquidite max, data Cloud gratuite). **Fenetre = 2020-01 -> 2025-12** : le segment le plus decisif (bear 2022 + momentum 2024-25, cf `M12_HF_PROPOSAL.md` §4.5), qui couvre aussi les listings SOL/DOT (2020).

In [ ]:
from QuantConnect.Research import QuantBook
from QuantConnect import Resolution, Market
from datetime import datetime
import numpy as np
import pandas as pd
import gc

qb = QuantBook()

COINS = ['BTCUSD', 'ETHUSD', 'SOLUSD', 'LTCUSD', 'XRPUSD', 'ADAUSD', 'DOTUSD']
START = datetime(2020, 1, 1)   # segment decisif 2020-2025 (bear 2022 + momentum 2024-25)
END   = datetime(2025, 12, 31)

symbols = {}
for ticker in COINS:
    try:
        symbols[ticker] = qb.AddCrypto(ticker, Resolution.Minute, Market.Binance).Symbol
    except Exception as e:
        print(f'{ticker}: AddCrypto failed -> {type(e).__name__}: {e}')

print(f'{len(symbols)}/{len(COINS)} coins ajoutes (Binance). Fenetre : {START.date()} -> {END.date()} (0 QCC, Cloud Access gratuit).')

## 2. Estimateurs frequency-agnostiques (RV / BPV / J)

Independants de la frequence : operent sur n'importe quelle serie de rendements intraday indexes par jour. Identiques a `scripts/realized_variance.py`.

$$RV_t = \sum_{h \in t} r_h^2 \qquad BV_t = \frac{\pi}{2} \sum_{i=2}^{M} |r_i|\,|r_{i-1}| \qquad J_t = \max(RV_t - \mu\,BV_t,\,0),\ \mu=0.6$$

In [ ]:
MU_HUANG_TAUCHEN = 0.6   # seuil de saut (Huang-Tauchen 2005)
MIN_OBS_PER_DAY  = 6    # drop sessions incompletes

def daily_realized_variance(r, min_obs=MIN_OBS_PER_DAY):
    sq = r.astype(float) ** 2
    g  = sq.groupby(sq.index.normalize())
    rv = g.sum()
    rv = rv[g.count() >= min_obs]
    rv.index = pd.DatetimeIndex(rv.index).normalize()
    return rv.rename('RV')

def daily_bipower_variation(r, min_obs=MIN_OBS_PER_DAY):
    a  = r.abs().astype(float)
    al = a.shift(1)
    same_day = a.index.normalize() == al.index.normalize()
    paired = (a * al).where(same_day, 0.0)
    g  = paired.groupby(paired.index.normalize())
    bv = (np.pi / 2.0) * g.sum()
    bv = bv[a.groupby(a.index.normalize()).count() >= min_obs]
    bv.index = pd.DatetimeIndex(bv.index).normalize()
    return bv.rename('BV')

def daily_jump_component(r, mu=MU_HUANG_TAUCHEN, min_obs=MIN_OBS_PER_DAY):
    rv = daily_realized_variance(r, min_obs)
    bv = daily_bipower_variation(r, min_obs)
    df = pd.concat([rv, bv], axis=1).dropna()
    j  = np.maximum(df['RV'] - mu * df['BV'], 0.0)
    return j.rename('J')

def daily_returns(r):
    d = r.groupby(r.index.normalize()).sum()
    d.index = pd.DatetimeIndex(d.index).normalize()
    return d.rename('r')

def har_lag_features(rv):
    rv = rv.astype(float)
    return pd.DataFrame({
        'rv_d': rv.shift(1),
        'rv_w': rv.shift(1).rolling(5, min_periods=5).mean(),
        'rv_m': rv.shift(1).rolling(22, min_periods=22).mean(),
    })

def to_log_rv(rv, eps=1e-12):
    return np.log(rv.astype(float).clip(lower=eps))

print('Estimateurs RV/BPV/J (frequency-agnostiques) definis.')

## 3. Pipeline per-coin incrémental + chunking annuel (anti-OOM)

La data minute (~4M bars/coin/an, ~24M sur 6 ans) ne tient PAS en memoire si tiree d'un bloc. On la tire **annee par annee**, on agrege en series journalieres (RV/BPV/J/rets ~2K lignes/an), puis on **libere** la minute avant l'annee suivante. La serie horaire (24 obs/j, ~50K bars/coin total) est tiree en un seul coup (negligeable). Resultat : `daily[coin][freq] = {rv, jumps, daily_rets}`, memoire bornee a ~1 an de minute a la fois.

In [ ]:
def intraday_log_returns(symbol, resolution, start, end):
    """Close-to-close log returns at `resolution`, tz-naive UTC DatetimeIndex."""
    hist = qb.History(symbol, start, end, resolution)
    if hist is None or len(hist) == 0:
        return pd.Series(dtype=float)
    if isinstance(hist.index, pd.MultiIndex):
        close = hist['close'].xs(symbol, level='symbol') if 'symbol' in hist.index.names else hist['close']
    else:
        close = hist['close']
    close = close.sort_index().astype(float)
    rets = np.log(close).diff().dropna()
    if rets.index.tz is not None:
        rets.index = rets.index.tz_convert('UTC').tz_localize(None)
    return rets.rename('r')

def _dedup_concat(parts):
    if not parts:
        return None
    s = pd.concat(parts).sort_index()
    return s[~s.index.duplicated(keep='last')]

def build_minute_daily(symbol, start, end):
    """Pull minute data YEAR BY YEAR (memory-bounded) -> {rv, jumps, daily_rets} or None."""
    rvs, jps, drs = [], [], []
    y = start.year
    while y <= end.year:
        ys = max(datetime(y, 1, 1), start); ye = min(datetime(y, 12, 31), end)
        rets = intraday_log_returns(symbol, Resolution.Minute, ys, ye)
        if len(rets) >= 1000:
            rvs.append(daily_realized_variance(rets))
            jps.append(daily_jump_component(rets))
            drs.append(daily_returns(rets))
        del rets
        y += 1
    rv, jp, dr = _dedup_concat(rvs), _dedup_concat(jps), _dedup_concat(drs)
    if rv is None or len(rv) < 200:
        return None
    return {'rv': rv, 'jumps': jp, 'daily_rets': dr}

def build_hour_daily(symbol, start, end):
    """Pull hourly data in ONE shot (~50K bars/coin, negligible) -> {rv, jumps, daily_rets} or None."""
    rets = intraday_log_returns(symbol, Resolution.Hour, start, end)
    if len(rets) < 1000:
        return None
    return {'rv': daily_realized_variance(rets), 'jumps': daily_jump_component(rets), 'daily_rets': daily_returns(rets)}

daily = {}  # daily[coin][freq] = {rv, jumps, daily_rets}
for ticker, sym in symbols.items():
    daily[ticker] = {}
    dmin = build_minute_daily(sym, START, END)
    gc.collect()
    dhr  = build_hour_daily(sym, START, END)
    gc.collect()
    daily[ticker]['minute'] = dmin
    daily[ticker]['hour']   = dhr
    nmin = len(dmin['rv']) if dmin else 0
    nhr  = len(dhr['rv'])  if dhr  else 0
    jmp  = int((dmin['jumps'] > 0).sum()) if dmin else 0
    print(f'{ticker:7s}: minute={nmin:>5}j (jumps>0={jmp}) | hour={nhr:>5}j')

print('\nPipeline per-coin termine. Data Cloud tiree (0 QCC).')

## 4. Modeles HAR (baseline) et HAR-RV-J, walk-forward 5-fold

Identiques a `scripts/m12_har_rv_j.py`. OLS sur log-RV, features HAR (d/w/m) [+ sauts d/w/m pour HAR-RV-J]. Walk-forward expanding, refit tous les 22 jours. Prevision iteree h-step.

In [ ]:
N_SPLITS  = 5
REFIT_EVERY = 22

def _har_features(rv):
    f = har_lag_features(rv)
    return f.apply(to_log_rv)

def _har_rvj_features(rv, jumps):
    logf = _har_features(rv)
    return pd.DataFrame({
        'rv_d': logf['rv_d'], 'rv_w': logf['rv_w'], 'rv_m': logf['rv_m'],
        'j_d': jumps.shift(1),
        'j_w': jumps.shift(1).rolling(5, min_periods=5).mean(),
        'j_m': jumps.shift(1).rolling(22, min_periods=22).mean(),
    })

class _OLSModel:
    def __init__(self, feat_fn, use_jumps):
        self.feat_fn = feat_fn; self.use_jumps = use_jumps; self.coef_ = None
    def fit(self, rv, jumps):
        feats = self.feat_fn(rv, jumps) if self.use_jumps else self.feat_fn(rv)
        y = to_log_rv(rv).rename('y')
        df = pd.concat([feats, y], axis=1).dropna()
        if len(df) < 30:
            raise ValueError(f'fit needs >=30 obs, got {len(df)}')
        X = np.column_stack([np.ones(len(df)), df.drop(columns='y').to_numpy()])
        self.coef_, *_ = np.linalg.lstsq(X, df['y'].to_numpy(), rcond=None)
        return self
    def predict_h_step(self, rv_hist, j_hist, horizon):
        rv_vals = list(rv_hist.astype(float).values)
        j_vals  = list(j_hist.astype(float).values) if j_hist is not None else None
        preds = []
        for _ in range(horizon):
            trv = pd.Series(rv_vals[-22:])
            log_rv = np.log(trv.clip(lower=1e-12))
            row = [1.0, float(log_rv.iloc[-1]), float(log_rv.iloc[-5:].mean()), float(log_rv.iloc[-22:].mean())]
            if self.use_jumps:
                tj = pd.Series(j_vals[-22:])
                row += [float(tj.iloc[-1]), float(tj.iloc[-5:].mean()), float(tj.iloc[-22:].mean())]
            lp = float(np.array(row) @ self.coef_)
            preds.append(lp)
            rv_vals.append(float(np.exp(lp)))
            if j_vals is not None:
                j_vals.append(0.0)
        return float(np.mean(preds))

def walk_forward(rv, jumps, horizon, use_jumps, n_splits=N_SPLITS, refit=REFIT_EVERY):
    rv = rv.dropna().astype(float)
    if use_jumps:
        jumps = jumps.dropna().astype(float)
        idx = rv.index.intersection(jumps.index); rv = rv.loc[idx]; jumps = jumps.loc[idx]
    n = len(rv)
    if n < 200:
        raise ValueError(f'need >=200 daily obs, got {n}')
    fold = n // (n_splits + 1)
    if fold < 30:
        raise ValueError(f'n={n} trop petit pour {n_splits} splits')
    feat_fn = _har_rvj_features if use_jumps else _har_features
    preds, dates = [], []
    for k in range(1, n_splits + 1):
        tr_end = fold * k; te_s = tr_end; te_e = min(tr_end + fold, n)
        rv_tr = rv.iloc[:tr_end]; j_tr = jumps.iloc[:tr_end] if use_jumps else None
        if len(rv_tr) < 60:
            continue
        model = _OLSModel(feat_fn, use_jumps).fit(rv_tr, j_tr)
        h_rv = list(rv.iloc[:te_s].values)
        h_j  = list(jumps.iloc[:te_s].values) if use_jumps else None
        for i in range(te_s, te_e - horizon):
            tr = pd.Series(h_rv[-(22 + horizon):])
            tj = pd.Series(h_j[-(22 + horizon):]) if use_jumps else None
            preds.append(model.predict_h_step(tr, tj, horizon))
            dates.append(rv.index[i])
            h_rv.append(float(rv.iloc[i]))
            if use_jumps:
                h_j.append(float(jumps.iloc[i]))
            if (i - te_s) % refit == 0 and i > te_s:
                model = _OLSModel(feat_fn, use_jumps).fit(rv.iloc[:i], jumps.iloc[:i] if use_jumps else None)
    return pd.Series(preds, index=pd.DatetimeIndex(dates), name='pred_logrv')

print('Modeles HAR / HAR-RV-J + walk-forward 5-fold (refit 22j) definis.')

## 5. Sizing Kelly + tests statistiques (DM direction-aware, sign-test, LW Sharpe-diff)

Kelly cap=1.0 (M11i a confirme cap=3.0 tue par Calmar). fee=50 bps (break-even HAR Kelly). Le **verdict decisif** = M12-HF (minute) **vs M12-horaire** (hour) sur le **delta-Sharpe** netto de frais, avec test du signe apparie + SE HAC Ledoit-Wolf (2008).

In [ ]:
KELLY_CAP  = 1.0
MU_WINDOW  = 60
FEE_BPS    = 50
SEEDS      = [0, 1, 7, 42]
HORIZONS   = [1, 5, 10]

def kelly_weights_returns(daily_rets, fc_logrv, mu_window=MU_WINDOW, cap=KELLY_CAP):
    fc = fc_logrv.reindex(daily_rets.index).dropna()
    dr = daily_rets.reindex(fc.index).dropna()
    if len(dr) < 50:
        return None
    rolling_fc = fc.rolling(mu_window, min_periods=mu_window // 2).mean()
    raw = np.exp(-0.5 * rolling_fc)
    w = (raw / raw.rolling(mu_window, min_periods=mu_window // 2).std()).clip(-cap, cap)
    w = w.replace([np.inf, -np.inf], np.nan).dropna()
    dr = dr.reindex(w.index)
    return w.values, dr

def net_at_fee(weights, rets, fee_bps=FEE_BPS):
    w = np.asarray(weights, dtype=float)
    r = np.asarray(rets, dtype=float)
    turnover = np.abs(np.diff(np.concatenate([[0.0], w])))
    cost = (fee_bps / 1e4) * turnover
    return r * w - cost

def sharpe_ann(returns):
    if len(returns) < 10:
        return float('nan')
    mu, sd = float(np.mean(returns)), float(np.std(returns, ddof=1))
    return (mu / sd) * np.sqrt(365) if sd > 1e-12 else float('nan')

def ledoit_wolf_sharpe_diff_se(rets_a, rets_b):
    a = np.asarray(rets_a, dtype=float); b = np.asarray(rets_b, dtype=float)
    n = min(len(a), len(b))
    if n < 30:
        return float('nan')
    a, b = a[:n], b[:n]
    d = a - b
    lag = int(np.floor(4 * (n / 100) ** (2 / 9)))
    d_dm = d - np.mean(d)
    omega = float(np.dot(d_dm, d_dm) / n)
    for k in range(1, lag + 1):
        wk = 1 - k / (lag + 1)
        omega += 2 * wk * float(np.dot(d_dm[k:], d_dm[:-k]) / n)
    return float(np.sqrt(omega / n)) if omega > 0 else float('nan')

def sign_test_pvalue(n_pos, n_total):
    from math import comb
    k, m = n_pos, n_total
    return sum(comb(m, i) for i in range(k, m + 1)) / (2 ** m)

print('Kelly cap=1.0, fee=50bps, LW/sign-test definis. Baseline decisive = M12-horaire.')

## 6. Sweep : 7 coins x 3 horizons x 4 seeds = 84 combos, M12-HF (minute) vs M12-horaire (hour)

Pour chaque combo (coin, horizon, seed), on evalue HAR-RV-J sur **les deux frequences** (minute = M12-HF, hour = baseline M12-horaire), on calcule le Sharpe netto de frais de chacun, puis le **delta-Sharpe M12-HF - M12-horaire**. Le verdict global = test du signe sur les deltas. **Critere d'acceptation** : M12-HF `BEATS_p005` M12-horaire sur **>=4/7 coins a h=1**.

In [ ]:
import time
t0 = time.time()
rows, skipped = [], []
total = len(symbols) * len(HORIZONS) * len(SEEDS)

def eval_freq(d, horizon):
    if d is None or len(d['rv']) < 300:
        return None
    try:
        return walk_forward(d['rv'], d['jumps'], horizon, use_jumps=True), d['daily_rets']
    except Exception:
        return None

for ticker in symbols:
    d_min = daily[ticker].get('minute'); d_hr = daily[ticker].get('hour')
    for h in HORIZONS:
        r_min = eval_freq(d_min, h); r_hr = eval_freq(d_hr, h)
        if r_min is None or r_hr is None:
            skipped.append((ticker, h, 'forecast fail / data <300')); continue
        fc_m, dr_m = r_min; fc_h, dr_h = r_hr
        for seed in SEEDS:
            np.random.seed(seed)
            wm = kelly_weights_returns(dr_m, fc_m)
            wh = kelly_weights_returns(dr_h, fc_h)
            if wm is None or wh is None:
                skipped.append((ticker, h, seed, 'kelly fail')); continue
            net_m, net_h = net_at_fee(*wm), net_at_fee(*wh)
            sh_m, sh_h = sharpe_ann(net_m), sharpe_ann(net_h)
            delta = sh_m - sh_h
            se = ledoit_wolf_sharpe_diff_se(net_m, net_h)
            rows.append({
                'coin': ticker, 'horizon': h, 'seed': seed,
                'sharpe_m12hf_minute': sh_m, 'sharpe_m12_hour': sh_h,
                'delta_sharpe_hf_minus_hour': delta, 'lw_se': se,
                't_stat': delta / se if (se == se and se > 1e-12) else float('nan'),
                'n_obs': len(net_m),
            })

elapsed = time.time() - t0
df = pd.DataFrame(rows)
print(f'Sweep: {len(df)}/{total} combos en {elapsed:.0f}s ({len(skipped)} skipped)')
if skipped:
    print('Skipped:', skipped[:5], '...' if len(skipped) > 5 else '')
df.head()

## 7. Verdict — M12-HF bat-il M12-horaire ?

Verdict honnete **BEATS / NO BEATS / INCONCLUSIVE** par combo et global. Le critere d'acceptation = `BEATS_p005` sur **>=4/7 coins a h=1**. Un NO BEATS clarifie est un **resultat utile** (ferme la question frequence d'echantillonnage).

In [ ]:
if len(df) == 0:
    print('Aucun combo calcule (verifier la cellule 6 / disponibilite donnee).')
else:
    n = len(df); n_pos = int((df['delta_sharpe_hf_minus_hour'] > 0).sum())
    win_rate = n_pos / n
    p_sign = sign_test_pvalue(n_pos, n)
    med_delta = df['delta_sharpe_hf_minus_hour'].median()
    verdict = 'BEATS' if (p_sign < 0.05 and win_rate >= 0.60) else ('INCONCLUSIVE' if (p_sign < 0.10 and win_rate >= 0.55) else 'NO BEATS')

    print('=== Verdict global M12-HF (minute) vs M12-horaire (hour) ===')
    print(f'Verdict               : {verdict}')
    print(f'Win rate              : {win_rate*100:.1f}% ({n_pos}/{n} combos)')
    print(f'Test du signe p-value : {p_sign:.4e}')
    print(f'Median delta-Sharpe   : {med_delta:+.4f}')

    print('\n=== Critere d\'acceptation : >=4/7 coins BEATS a h=1 ===')
    h1 = df[df['horizon'] == 1]
    coin_wins = {}
    for c in h1['coin'].unique():
        sub = h1[h1['coin'] == c]
        cw = int((sub['delta_sharpe_hf_minus_hour'] > 0).sum())
        coin_wins[c] = (cw, len(sub), sub['delta_sharpe_hf_minus_hour'].median())
    n_beats_h1 = sum(1 for c, (w, t, m) in coin_wins.items() if w > t / 2)
    for c, (w, t, m) in sorted(coin_wins.items()):
        flag = '  BEATS' if w > t / 2 else ''
        print(f'  {c:7s}: {w}/{t} win, median delta {m:+.4f}{flag}')
    accept = n_beats_h1 >= 4
    print(f'\nCoins BEATS a h=1 : {n_beats_h1}/7 -> acceptation (>=4/7) = {accept}')

    print('\n=== Per-horizon ===')
    for h in HORIZONS:
        sub = df[df['horizon'] == h]
        w = int((sub['delta_sharpe_hf_minus_hour'] > 0).sum())
        print(f'  h={h}: {w}/{len(sub)} win, median {sub["delta_sharpe_hf_minus_hour"].median():+.4f}')

    print(f'\nVERDICT FINAL: {verdict} (p={p_sign:.2e}, win={win_rate*100:.1f}%, h=1 coins BEATS={n_beats_h1}/7)')

## 8. Conclusion

- **BEATS** : monter en frequence (minute) resserre l'estimateur de realized variance et nettoie le signal de saut -> M12-HF remplace M12-horaire comme KEEPER vol. Gain attendu concentre sur h=1 + le terme de saut.
- **NO BEATS** : le bruit de microstructure minute annule le gain de precision (Hansen-Lunde), OU la fenetre alignee (2020-2025) ne laisse rien a gagnerer. M12-horaire reste le KEEPER ; M12-HF clos negatif = **resultat utile** (la frequence d'echantillonnage ne plafonne pas M12).
- **INCONCLUSIVE** : signal marginal, a trancher par une etude per-coin de la noise-to-signal ratio (sparse sampling 5-min) en cycle suivant.

### References
- Andersen, Bollerslev, Diebold & Labys (2003). *Modeling and Forecasting Realized Volatility*, Econometrica 71(2):579-625.
- Andersen, Bollerslev & Diebold (2007). *Roughing It Up*, RESTAT 89(4):701-720.
- Barndorff-Nielsen & Shephard (2002). *Econometric Analysis of Realized Volatility...* (bipower variation).
- Hansen & Lunde (2006). *Realized Variance and Market Microstructure Noise*, JBES 24(2):127-161.
- Huang & Tauchen (2005). seuil $\mu \approx 0.6$.

Docs internes : [`M12_HF_PROPOSAL.md`](docs/M12_HF_PROPOSAL.md), [`M12_HAR_RV_J.md`](docs/M12_HAR_RV_J.md) (resultats M12-horaire).